# Compile Time-Averaged Planform Metrics

Reads the per-year sinuosity, CCI, and CFI values produced by `Planform_calculator_adapted_for_cleaned_subfolder.ipynb` (stored in `{river}_metrics.xlsx` files) and compiles time-averaged values for each reach into a single output CSV.

**Input CSV columns required** (same format as the planform calculator):
- `river_name`
- `working_directory`
(all other columns from the planform calculator CSV are accepted but not used here)

**Output CSV columns:**
- `river_name`
- `ds_order` — reach number (integer extracted from `reach_X` sheet name)
- `sinuosity` — mean sinuosity across all available years
- `CCI` — mean Channel Count Index across all available years
- `CFI` — mean Channel Form Index across all available years

The output CSV is saved in the same directory as the input CSV.

Author: James (Huck) Rees; PhD Student, UCSB Geography

## Import packages

In [4]:
import os
import re
import pandas as pd

## Core function

In [5]:
def compile_planform_metrics(input_csv_path):
    """
    Reads per-year planform metrics from {river}_metrics.xlsx files and compiles
    time-averaged sinuosity, CCI, and CFI for each reach into a single output CSV.

    Parameters
    ----------
    input_csv_path : str
        Path to the input CSV (same format used by the planform calculator).
        Must contain columns: river_name, working_directory.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns: river_name, ds_order, sinuosity, CCI, CFI.
        Also saved as a CSV next to the input CSV.
    """
    river_data = pd.read_csv(input_csv_path)
    required_cols = {'river_name', 'working_directory'}
    missing = required_cols - set(river_data.columns)
    if missing:
        raise ValueError(f"Input CSV is missing required columns: {missing}")

    rows = []

    for _, row in river_data.iterrows():
        river_name = row['river_name']
        working_directory = row['working_directory']

        metrics_path = os.path.join(
            working_directory, "RiverMapping", "Channels",
            river_name, f"{river_name}_metrics.xlsx"
        )

        if not os.path.exists(metrics_path):
            print(f"  [SKIP] Metrics file not found for {river_name}: {metrics_path}")
            continue

        print(f"Processing {river_name} ...")

        xl = pd.ExcelFile(metrics_path)

        for sheet_name in xl.sheet_names:
            # Sheet names are expected to be 'reach_1', 'reach_2', etc.
            match = re.match(r"reach_(\d+)", sheet_name)
            if not match:
                print(f"  [SKIP] Unrecognised sheet name '{sheet_name}' in {river_name} — expected 'reach_N'")
                continue

            ds_order = int(match.group(1))

            df = xl.parse(sheet_name)

            # Expected columns (case-insensitive lookup for robustness)
            col_map = {c.lower(): c for c in df.columns}
            needed = {'sinuosity', 'cci', 'cfi'}
            missing_cols = needed - set(col_map.keys())
            if missing_cols:
                print(f"  [SKIP] Sheet '{sheet_name}' for {river_name} missing columns: {missing_cols}")
                continue

            mean_sinuosity = df[col_map['sinuosity']].mean(skipna=True)
            mean_cci      = df[col_map['cci']].mean(skipna=True)
            mean_cfi      = df[col_map['cfi']].mean(skipna=True)

            n_years = df[col_map['sinuosity']].count()
            print(f"  reach {ds_order}: {n_years} year(s) averaged — "
                  f"sinuosity={mean_sinuosity:.4f}, CCI={mean_cci:.4f}, CFI={mean_cfi:.4f}")

            rows.append({
                'river_name': river_name,
                'ds_order':   ds_order,
                'sinuosity':  mean_sinuosity,
                'CCI':        mean_cci,
                'CFI':        mean_cfi,
            })

    if not rows:
        print("No data compiled. Check that metrics Excel files exist at the expected paths.")
        return pd.DataFrame(columns=['river_name', 'ds_order', 'sinuosity', 'CCI', 'CFI'])

    result_df = pd.DataFrame(rows).sort_values(['river_name', 'ds_order']).reset_index(drop=True)

    # Save next to the input CSV
    output_dir = os.path.dirname(os.path.abspath(input_csv_path))
    output_path = os.path.join(output_dir, 'planform_metrics_compiled.csv')
    result_df.to_csv(output_path, index=False)
    print(f"\nOutput saved to: {output_path}")

    return result_df

## Run

In [6]:
csv_path = r"E:\Dissertation\Data\Zhaoetal2025_river_datasheet_og.csv"
result = compile_planform_metrics(csv_path)
result

Processing Amazonas_Jatuarana ...
  reach 1: 36 year(s) averaged — sinuosity=1.1713, CCI=4.3750, CFI=0.2771

Output saved to: E:\Dissertation\Data\planform_metrics_compiled.csv


,river_name,ds_order,sinuosity,CCI,CFI
0,Amazonas_Jatuarana,1,1.171287,4.375,0.277088
